In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from thesis.common.logger import setup_logger
from thesis.eta.experiment import convert_results_to_dataframe, initialize_experiment, load_experiment_results

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

experiment_name = "comparison"
_, logs_dir, _, _ = initialize_experiment(experiment_name)
logger = setup_logger(experiment_name, logs_dir)

In [ ]:
all_results = load_experiment_results()
df_results = convert_results_to_dataframe(all_results)

## Compare Experiments to Baseline

Let's compare all experiments to the baseline performance using MAPE and MAE as our primary metrics.


In [ ]:
# Use all models (including linear regression as baseline)
df_all = df_results.copy()

# Calculate relative improvements compared to baseline
baseline_results = df_all[df_all["experiment"] == "baseline"].set_index(["scenario", "model"])

# For each experiment, calculate the improvement over baseline
for experiment in df_all["experiment"].unique():
    if experiment != "baseline":
        exp_results = df_all[df_all["experiment"] == experiment].set_index(["scenario", "model"])

        # Calculate relative change in MAPE (lower is better, so negative is improvement)
        df_all.loc[df_all["experiment"] == experiment, "mape_improvement"] = df_all[
            df_all["experiment"] == experiment
        ].apply(
            lambda row: (baseline_results.loc[(row["scenario"], row["model"]), "mape"] - row["mape"])
            / baseline_results.loc[(row["scenario"], row["model"]), "mape"]
            * 100,
            axis=1,
        )

        # Calculate relative change in MAE (lower is better, so negative is improvement)
        df_all.loc[df_all["experiment"] == experiment, "mae_improvement"] = df_all[
            df_all["experiment"] == experiment
        ].apply(
            lambda row: (baseline_results.loc[(row["scenario"], row["model"]), "mae"] - row["mae"])
            / baseline_results.loc[(row["scenario"], row["model"]), "mae"]
            * 100,
            axis=1,
        )

# Display improvements
improvement_cols = ["experiment", "scenario", "model", "mape", "mae", "mape_improvement", "mae_improvement"]
df_improvements = df_all[df_all["experiment"] != "baseline"][improvement_cols].round(4)
df_improvements.sort_values(["mape_improvement"], ascending=False).head(10)

## Average Performance by Experiment


In [ ]:
# Calculate average metrics across all scenarios and models for each experiment
avg_by_experiment = df_all.groupby("experiment")[["mape", "mae", "rmse"]].mean().round(4)

# Add improvement columns for non-baseline experiments
for experiment in avg_by_experiment.index:
    if experiment != "baseline":
        improvement_data = df_all[df_all["experiment"] == experiment][["mape_improvement", "mae_improvement"]].mean()
        avg_by_experiment.loc[experiment, "avg_mape_improvement"] = improvement_data["mape_improvement"]
        avg_by_experiment.loc[experiment, "avg_mae_improvement"] = improvement_data["mae_improvement"]

# Sort by MAPE (lower is better)
avg_by_experiment = avg_by_experiment.sort_values("mape")
print("Average Performance by Experiment (All Models Including Linear Regression):")
print("=" * 80)
avg_by_experiment


## Performance by Model


In [ ]:
# Average performance by model across all experiments
avg_by_model = df_all.groupby("model")[["mape", "mae", "rmse"]].mean().round(4)
avg_by_model = avg_by_model.sort_values("mape")

print("Average Performance by Model (Across All Experiments):")
print("=" * 60)
avg_by_model

## Performance by Scenario


In [ ]:
# Performance by scenario
avg_by_scenario = df_all.groupby("scenario")[["mape", "mae", "rmse"]].mean().round(4)
avg_by_scenario = avg_by_scenario.sort_values("mape")

print("Average Performance by Scenario (All Models):")
print("=" * 60)
avg_by_scenario


## Overall Rankings

Let's create a comprehensive ranking of all experiment-model-scenario combinations.


In [ ]:
# Create a ranking based on MAPE (lower is better)
df_ranking = df_all[["experiment", "scenario", "model", "mape", "mae", "rmse"]].copy()
df_ranking["rank_by_mape"] = df_ranking["mape"].rank()
df_ranking["rank_by_mae"] = df_ranking["mae"].rank()  # Lower MAE is better

# Sort by MAPE rank
df_ranking = df_ranking.sort_values("rank_by_mape")

# Show top 15 best performing combinations
print("Top 15 Best Performing Combinations (by MAPE):")
print("=" * 100)
df_ranking.head(15)


## Visualizations


In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. MAPE by Experiment
ax1 = axes[0, 0]
mape_by_exp = df_all.groupby("experiment")["mape"].mean().sort_values()
mape_by_exp.plot(kind="barh", ax=ax1, color="skyblue")
ax1.set_xlabel("Average MAPE")
ax1.set_title("Average MAPE by Experiment (Lower is Better)")
ax1.grid(True, alpha=0.3)

# 2. MAE by Experiment
ax2 = axes[0, 1]
mae_by_exp = df_all.groupby("experiment")["mae"].mean().sort_values()
mae_by_exp.plot(kind="barh", ax=ax2, color="lightgreen")
ax2.set_xlabel("Average MAE")
ax2.set_title("Average MAE by Experiment (Lower is Better)")
ax2.grid(True, alpha=0.3)

# 3. MAPE by Model
ax3 = axes[1, 0]
mape_by_model = df_all.groupby("model")["mape"].mean().sort_values()
mape_by_model.plot(kind="barh", ax=ax3, color="salmon")
ax3.set_xlabel("Average MAPE")
ax3.set_title("Average MAPE by Model")
ax3.grid(True, alpha=0.3)

# 4. MAPE by Scenario
ax4 = axes[1, 1]
mape_by_scenario = df_all.groupby("scenario")["mape"].mean().sort_values()
mape_by_scenario.plot(kind="barh", ax=ax4, color="gold")
ax4.set_xlabel("Average MAPE")
ax4.set_title("Average MAPE by Scenario")
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Create a heatmap of MAPE values
plt.figure(figsize=(14, 8))

# Pivot the data for heatmap
pivot_data = df_all.pivot_table(values="mape", index="experiment", columns=["scenario", "model"], aggfunc="mean")

# Create heatmap
sns.heatmap(pivot_data, annot=True, fmt=".4f", cmap="RdYlGn_r", cbar_kws={"label": "MAPE"}, linewidths=0.5)
plt.title("MAPE Heatmap: Experiments vs Scenarios/Models (Including Linear Regression)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Create a heatmap of MAE values
plt.figure(figsize=(14, 8))

# Pivot the data for heatmap
pivot_data = df_all.pivot_table(values="mae", index="experiment", columns=["scenario", "model"], aggfunc="mean")

# Create heatmap
sns.heatmap(pivot_data, annot=True, fmt=".3f", cmap="RdYlGn_r", cbar_kws={"label": "MAE"}, linewidths=0.5)
plt.title("MAE Heatmap: Experiments vs Scenarios/Models (Including Linear Regression)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Summary and Insights


In [ ]:
# Find the best experiment overall (excluding baseline)
best_experiments = df_all[df_all["experiment"] != "baseline"].groupby("experiment")[["mape", "mae"]].mean()
best_experiment = best_experiments["mape"].idxmin()
best_mape = best_experiments.loc[best_experiment, "mape"]
best_mae = best_experiments.loc[best_experiment, "mae"]

# Calculate baseline averages
baseline_avg = df_all[df_all["experiment"] == "baseline"][["mape", "mae"]].mean()

print("SUMMARY OF EXPERIMENTS")
print("=" * 80)
print(f"\nBest Performing Experiment: {best_experiment}")
print(f"  - Average MAPE: {best_mape:.4f} (Baseline: {baseline_avg['mape']:.4f})")
print(f"  - Average MAE: {best_mae:.4f} (Baseline: {baseline_avg['mae']:.4f})")
print(f"  - MAPE Improvement: {((baseline_avg['mape'] - best_mape) / baseline_avg['mape'] * 100):.2f}%")
print(f"  - MAE Improvement: {((baseline_avg['mae'] - best_mae) / baseline_avg['mae'] * 100):.2f}%")

# Best model overall
best_model_avg = df_all.groupby("model")[["mape", "mae"]].mean()
best_model = best_model_avg["mape"].idxmin()
print(f"\nBest Performing Model Overall: {best_model}")
print(f"  - Average MAPE: {best_model_avg.loc[best_model, 'mape']:.4f}")
print(f"  - Average MAE: {best_model_avg.loc[best_model, 'mae']:.4f}")

# Scenario difficulty ranking
print("\nScenario Difficulty (by average MAPE):")
for scenario in avg_by_scenario.index:
    print(
        f"  - {scenario}: MAPE = {avg_by_scenario.loc[scenario, 'mape']:.4f}, MAE = {avg_by_scenario.loc[scenario, 'mae']:.4f}"
    )

# Top 5 best combinations
print("\nTop 5 Best Performing Combinations:")
for idx, row in df_ranking.head(5).iterrows():
    print(
        f"  {int(row['rank_by_mape'])}. {row['experiment']} + {row['model']} + {row['scenario']}: MAPE = {row['mape']:.4f}, MAE = {row['mae']:.4f}"
    )


In [ ]:
# Create improvement visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Calculate average improvements for each experiment
improvements = (
    df_all[df_all["experiment"] != "baseline"].groupby("experiment")[["mape_improvement", "mae_improvement"]].mean()
)

# MAPE Improvement
improvements["mape_improvement"].sort_values(ascending=False).plot(kind="bar", ax=ax1, color="steelblue")
ax1.set_ylabel("MAPE Improvement (%)")
ax1.set_title("Average MAPE Improvement over Baseline")
ax1.axhline(y=0, color="red", linestyle="--", alpha=0.5)
ax1.grid(True, alpha=0.3)
ax1.set_xlabel("Experiment")

# MAE Improvement
improvements["mae_improvement"].sort_values(ascending=False).plot(kind="bar", ax=ax2, color="forestgreen")
ax2.set_ylabel("MAE Improvement (%)")
ax2.set_title("Average MAE Improvement over Baseline")
ax2.axhline(y=0, color="red", linestyle="--", alpha=0.5)
ax2.grid(True, alpha=0.3)
ax2.set_xlabel("Experiment")

plt.tight_layout()
plt.show()